### 金融异常检测任务

## 1. 实验介绍

反欺诈是金融行业永恒的主题，在互联网金融信贷业务中，数字金融反欺诈技术已经得到广泛应用并取得良好效果，这其中包括了近几年迅速发展并在各个领域
得到越来越广泛应用的神经网络。本项目以互联网智能风控为背景，从用户相互关联和影响的视角，探索满足风控反欺诈领域需求的，可拓展、高效的神经
网络应用方案，从而帮助更好地识别欺诈用户。

本项目主要关于实现预测模型(**项目用图神经网络举例，具体实现可以使用其他模型**)，进行节点异常检测任务，并验证模型精度。而本项目基于的数据集[DGraph](https://dgraph.xinye.com/introduction)，[DGraph](https://dgraph.xinye.com/introduction)
是大规模动态图数据集的集合，由真实金融场景中随着时间演变事件和标签构成。

### 1.1 实验目的

- 了解如何使用Pytorch进行神经网络训练
- 了解如何使用Pytorch-geometric等图网络深度学习库进行简单图神经网络设计(推荐使用GAT, GraphSAGE模型)。
- 了解如何利用MO平台进行模型性能评估。

### 1.2 预备知识
- 具备一定的深度学习理论知识，如卷积神经网络、损失函数、优化器，训练策略等。
- 了解并熟悉Pytorch计算框架。
- 学习Pytorch-geometric，请前往：https://pytorch-geometric.readthedocs.io/en/latest/
    
### 1.3实验环境
- numpy = 1.26.4  
- pytorch = 2.3.1  
- torch_geometric = 2.5.3  
- torch_scatter = 2.1.2  
- torch_sparse = 0.6.18  

## 2. 实验内容

### 2.1 数据集信息
DGraph-Fin 是一个由数百万个节点和边组成的有向无边权的动态图。它代表了Finvolution Group用户之间的社交网络，其中一个节点对应一个Finvolution 用户，从一个用户到另一个用户的边表示**该用户将另一个用户视为紧急联系人**。
下面是`位于dataset/DGraphFin目录`的DGraphFin数据集的描述:
```
x:  20维节点特征向量
y:  节点对应标签，一共包含四类。其中类1代表欺诈用户而类0代表正常用户(实验中需要进行预测的两类标签)，类2和类3则是背景用户，即无需预测其标签。
edge_index:  图数据边集,每条边的形式(id_a,id_b)，其中ids是x中的索引
edge_type: 共11种类型的边
edge_timestamp: 脱敏后的时间戳
train_mask, valid_mask, test_mask: 训练集，验证集和测试集掩码
```
本预测任务为识别欺诈用户的节点预测任务,只需要将欺诈用户（Class 1）从正常用户（Class 0）中区分出来。需要注意的是，其中测试集中样本对应的label**均被标记为-100**。

### 2.2 导入相关包

导入相应模块，设置数据集路径、设备等。

In [1]:
from utils import DGraphFin
from utils.utils import prepare_folder
from utils.evaluator import Evaluator

import torch
import torch.nn.functional as F
import torch.nn as nn

import torch_geometric.transforms as T

import numpy as np
from torch_geometric.data import Data
import os

#设置gpu设备
device = 0
device = f'cuda:{device}' if torch.cuda.is_available() else 'cpu'
device = torch.device(device)


### 2.3 数据处理

在使用数据集训练网络前，首先需要对数据进行归一化等预处理，如下：

In [2]:
path='./datasets/632d74d4e2843a53167ee9a1-momodel/' #数据保存路径
save_dir='./results/' #模型保存路径
dataset_name='DGraph'
dataset = DGraphFin(root=path, name=dataset_name, transform=T.ToSparseTensor())

nlabels = dataset.num_classes
if dataset_name in ['DGraph']:
    nlabels = 2    #本实验中仅需预测类0和类1

data = dataset[0]
print(data)
data.adj_t = data.adj_t.to_symmetric() #将有向图转化为无向图


if dataset_name in ['DGraph']:
    x = data.x
    x = (x - x.mean(0)) / x.std(0)
    data.x = x
if data.y.dim() == 2:
    data.y = data.y.squeeze(1)

split_idx = {'train': data.train_mask, 'valid': data.valid_mask, 'test': data.test_mask}  #划分训练集，验证集

train_idx = split_idx['train']
result_dir = prepare_folder(dataset_name,'mlp')

Data(x=[3700550, 20], edge_attr=[4300999], y=[3700550, 1], train_mask=[857899], valid_mask=[183862], test_mask=[183840], adj_t=[3700550, 3700550, nnz=4300999])


这里我们可以查看数据各部分维度

In [12]:
print(data)
print(data.x.shape)  #feature
print(data.y.shape)  #label

Data(x=[3700550, 20], edge_attr=[4300999], y=[3700550], train_mask=[857899], valid_mask=[183862], test_mask=[183840], adj_t=[3700550, 3700550, nnz=7994520])
torch.Size([3700550, 20])
torch.Size([3700550])


### 2.4 定义模型
这里我们使用简单的多层感知机作为例子：

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self
                 , in_channels
                 , hidden_channels
                 , out_channels
                 , num_layers
                 , dropout
                 , batchnorm=True):
        super(MLP, self).__init__()
        self.lins = torch.nn.ModuleList()
        self.lins.append(torch.nn.Linear(in_channels, hidden_channels))
        self.batchnorm = batchnorm
        if self.batchnorm:
            self.bns = torch.nn.ModuleList()
            self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.lins.append(torch.nn.Linear(hidden_channels, hidden_channels))
            if self.batchnorm:
                self.bns.append(torch.nn.BatchNorm1d(hidden_channels))
        self.lins.append(torch.nn.Linear(hidden_channels, out_channels))

        self.dropout = dropout

    def reset_parameters(self):
        for lin in self.lins:
            lin.reset_parameters()
        if self.batchnorm:
            for bn in self.bns:
                bn.reset_parameters()

    def forward(self, x):
        for i, lin in enumerate(self.lins[:-1]):
            x = lin(x)
            if self.batchnorm:
                x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lins[-1](x)
        return F.log_softmax(x, dim=-1)


配置后续训练、验证、推理用到的参数。可以调整以下超参以提高模型训练后的验证精度：

- `epochs`：在训练集上训练的代数；
- `lr`：学习率；
- `num_layers`：网络的层数；
- `hidden_channels`：隐藏层维数；
- `dropout`：dropout比例；
- `weight_decay`：正则化项的系数。

In [ ]:
mlp_parameters = {
    'lr': 0.01
    , 'num_layers': 2
    , 'hidden_channels': 128
    , 'dropout': 0.0
    , 'batchnorm': False
    , 'weight_decay': 5e-7
                  }
epochs = 200
log_steps =10 # log记录周期


初始化模型，并使用**Area Under the Curve (AUC)** 作为模型评价指标来衡量模型的表现。AUC通过对ROC曲线下各部分的面积求和而得。

具体计算过程参见 https://github.com/scikit-learn/scikit-learn/blob/baf828ca1/sklearn/metrics/_ranking.py#L363

In [ ]:
para_dict = mlp_parameters
model_para = mlp_parameters.copy()
model_para.pop('lr')
model_para.pop('weight_decay')
model = MLP(in_channels=data.x.size(-1), out_channels=nlabels, **model_para).to(device)
print(f'Model MLP initialized')


eval_metric = 'auc'  #使用AUC衡量指标
evaluator = Evaluator(eval_metric)


### 2.5 训练

使用训练集中的节点用于训练模型，并使用验证集进行挑选模型。

In [ ]:
def train(model, data, train_idx, optimizer):
     # data.y is labels of shape (N, )
    model.train()

    optimizer.zero_grad()

    out = model(data.x[train_idx])

    loss = F.nll_loss(out, data.y[train_idx])
    loss.backward()
    optimizer.step()

    return loss.item()


In [ ]:
def test(model, data, split_idx, evaluator):
    # data.y is labels of shape (N, )
    with torch.no_grad():
        model.eval()

        losses, eval_results = dict(), dict()
        for key in ['train', 'valid']:
            node_id = split_idx[key]

            out = model(data.x[node_id])
            y_pred = out.exp()  # (N,num_classes)

            losses[key] = F.nll_loss(out, data.y[node_id]).item()
            eval_results[key] = evaluator.eval(data.y[node_id], y_pred)[eval_metric]

    return eval_results, losses, y_pred


In [ ]:
print(sum(p.numel() for p in model.parameters()))  #模型总参数量

model.reset_parameters()
optimizer = torch.optim.Adam(model.parameters(), lr=para_dict['lr'], weight_decay=para_dict['weight_decay'])
best_valid = 0
min_valid_loss = 1e8

for epoch in range(1,epochs + 1):
    loss = train(model, data, train_idx, optimizer)
    eval_results, losses, out = test(model, data, split_idx, evaluator)
    train_eval, valid_eval = eval_results['train'], eval_results['valid']
    train_loss, valid_loss = losses['train'], losses['valid']

    if valid_loss < min_valid_loss:
        min_valid_loss = valid_loss
        torch.save(model.state_dict(), save_dir+'/model.pt') #将表现最好的模型保存

    if epoch % log_steps == 0:
        print(f'Epoch: {epoch:02d}, '
              f'Loss: {loss:.4f}, '
              f'Train: {100 * train_eval:.3f}, ' # 我们将AUC值乘上100，使其在0-100的区间内
              f'Valid: {100 * valid_eval:.3f} ')


### 2.6 模型预测

In [ ]:
model.load_state_dict(torch.load(save_dir+'/model.pt')) #载入验证集上表现最好的模型
def predict(data,node_id):
    """
    加载模型和模型预测
    :param node_id: int, 需要进行预测节点的下标
    :return: tensor, 类0以及类1的概率, torch.size[1,2]
    """
    # -------------------------- 实现模型预测部分的代码 ---------------------------
    with torch.no_grad():
        model.eval()
        out = model(data.x[node_id])
        y_pred = out.exp()  # (N,num_classes)

    return y_pred


In [ ]:
dic={0:"正常用户",1:"欺诈用户"}
node_idx = 0
y_pred = predict(data, node_idx)
print(y_pred)
print(f'节点 {node_idx} 预测对应的标签为:{torch.argmax(y_pred)}, 为{dic[torch.argmax(y_pred).item()]}。')

node_idx = 1
y_pred = predict(data, node_idx)
print(y_pred)
print(f'节点 {node_idx} 预测对应的标签为:{torch.argmax(y_pred)}, 为{dic[torch.argmax(y_pred).item()]}。')


## 3. 作业评分

**作业要求**：    
                         
1. 请加载你认为训练最佳的模型（不限于图神经网络)
2. 提交的作业包括【程序报告.pdf】和代码文件。

**注意：**
          
1. 在训练模型等过程中如果需要**保存数据、模型**等请写到 **results** 文件夹，如果采用 [离线任务](https://momodel.cn/docs/#/zh-cn/%E5%9C%A8GPU%E6%88%96CPU%E8%B5%84%E6%BA%90%E4%B8%8A%E8%AE%AD%E7%BB%83%E6%9C%BA%E5%99%A8%E5%AD%A6%E4%B9%A0%E6%A8%A1%E5%9E%8B) 请务必将模型保存在 **results** 文件夹下。
2. 训练出自己最好的模型后，先按照下列 cell 操作方式实现 NoteBook 加载模型测试；请测试通过在进行【系统测试】。
3. 点击左侧栏`提交作业`后点击`生成文件`则只需勾选 `predict()` 函数的cell，即【**模型预测代码答题区域**】的 cell。
4. 请导入必要的包和第三方库 (包括此文件中曾经导入过的)。
5. 请加载你认为训练最佳的模型，即请按要求填写**模型路径**。
6. `predict()`函数的输入和输出请不要改动。

===========================================  **模型预测代码答题区域**  =========================================== 

在下方的代码块中编写 **模型预测** 部分的代码，请勿在别的位置作答

In [3]:
import torch
import random
from torch import nn
import torch.nn.functional as F

class SageLayer(nn.Module):
    def __init__(self, input_size, out_size, gcn=False):
        super(SageLayer, self).__init__()
        self.input_size = input_size
        self.out_size = out_size
        self.gcn = gcn
        self.weight = nn.Parameter(torch.FloatTensor(out_size, input_size * (1 if gcn else 2)))
        self.init_params()
    
    def init_params(self):
        for param in self.parameters():
            nn.init.xavier_uniform_(param)
        
    def forward(self, self_feats, aggregate_feats, neighs=None):
        if not self.gcn:
            combined = torch.cat([self_feats, aggregate_feats], dim=1)
        else:
            combined = aggregate_feats
        combined = F.relu(self.weight.mm(combined.t())).t()
        return combined
    
class Classification(nn.Module):
    def __init__(self, embedding_size, num_classes):
        super(Classification, self).__init__()
        self.layer = nn.Sequential(nn.Linear(embedding_size, num_classes))
        self.init_params()
    
    def init_params(self):
        for param in self.parameters():
            if len(param.size()) == 2:
                nn.init.xavier_uniform_(param)
    
    def forward(self, embeds):
        logists = torch.log_softmax(self.layer(embeds), 1)
        return logists

In [4]:
class GraphSage(nn.Module):
    def __init__(
        self,
        num_layers,
        input_size,
        out_size,
        raw_features,
        adj_lists,
        device,
        num_classes=2,
        gcn=False,
        agg_func="mean",
    ):
        super(GraphSage, self).__init__()
        self.input_size = input_size
        self.out_size = out_size
        self.num_layers = num_layers
        self.raw_features = raw_features
        self.gcn = gcn
        self.device = device
        self.agg_func = agg_func
        self.adj_lists = adj_lists
        self.classifier = Classification(out_size, num_classes)
        layers = []
        
        for index in range(1, num_layers+1):
            layer_size = out_size if index != 1 else input_size
            layers.append(SageLayer(layer_size, out_size, self.gcn))
        layers.append(self.classifier)
        self.layers = nn.ModuleList(layers)
        
    def forward(self, nodes_batch):
        lower_layer_nodes = list(nodes_batch) #把当前训练的结点转化成list
        nodes_batch_layers = [(lower_layer_nodes,)]
        for i in range(self.num_layers):
            lower_samp_neighs, lower_layer_nodes_dict, lower_layer_nodes = self._get_unique_neighs_list(lower_layer_nodes)
            nodes_batch_layers.insert(0, (lower_layer_nodes, lower_samp_neighs, lower_layer_nodes_dict))
        pre_hidden_embs = self.raw_features
        for index in range(1, self.num_layers + 1):
            nb = nodes_batch_layers[index][0]
            pre_neighs = nodes_batch_layers[index-1]
            aggregate_feats = self.aggregate(nb, pre_hidden_embs, pre_neighs)
            sage_layer = self.layers[index - 1]
            if index > 1:
                nb = self._nodes_map(nb, pre_hidden_embs, pre_neighs)
            cur_hidden_embs = sage_layer(self_feats=pre_hidden_embs[nb], aggregate_feats=aggregate_feats)
            pre_hidden_embs = cur_hidden_embs
        return self.layers[-1](pre_hidden_embs)
    
    def _nodes_map(self, nodes, hidden_embs, neighs):
        layer_nodes, samp_neighs, layer_nodes_dict = neighs
        assert len(samp_neighs) == len(nodes)
        index = [layer_nodes_dict[x] for x in nodes]                    # 记录将上一层的节点编号。
        return index
    
    def _get_unique_neighs_list(self, nodes, num_sample=10):
        """
        从 SparseTensor (self.adj_lists) 高效采样邻居，复杂度 O(B × d)
        """
        adj_t = self.adj_lists
        if hasattr(nodes, "tolist"):
            nodes = nodes.tolist()

        # === 用 CSR 索引取邻居 ===
        # rowptr[i]: 第 i 行（节点）的邻居起始位置
        # col[rowptr[i]:rowptr[i+1]]: 第 i 行的所有邻居
        rowptr, col, _ = adj_t.csr()

        samp_neighs = []
        for node in nodes:
            start = rowptr[node].item()
            end = rowptr[node + 1].item()
            neigh = col[start:end].tolist()

            if len(neigh) == 0:
                s = set()
            elif num_sample is None:
                s = set(neigh)
            elif len(neigh) >= num_sample:
                s = set(random.sample(neigh, num_sample))
            else:
                s = set(random.choices(neigh, k=num_sample))

            # 包含自身
            s.add(node)
            samp_neighs.append(s)

        # 取并集得到涉及到的所有节点
        unique_nodes_list = list(set().union(*samp_neighs))
        unique_nodes = {nid: idx for idx, nid in enumerate(unique_nodes_list)}

        return samp_neighs, unique_nodes, unique_nodes_list

    def aggregate(self, nodes, pre_hidden_embs, pre_neighs, num_sample=10):
        unique_nodes_list, samp_neighs, unique_nodes = pre_neighs        # batch涉及到的所有节点,本身+邻居,邻居节点编号->字典中编号  
        assert len(nodes) == len(samp_neighs)
        indicator = [(nodes[i] in samp_neighs[i]) for i in range(len(samp_neighs))]  # 是否包含本身
        assert (False not in indicator)
        if not self.gcn:
            samp_neighs = [(samp_neighs[i]-set([nodes[i]])) for i in range(len(samp_neighs))]  # 在把中心节点去掉
        if len(pre_hidden_embs) == len(unique_nodes):                     # 保留需要使用的节点特征。
            embed_matrix = pre_hidden_embs
        else:
            embed_matrix = pre_hidden_embs[torch.LongTensor(unique_nodes_list)]                                               
        mask = torch.zeros(len(samp_neighs), len(unique_nodes))           # (本层节点数量，邻居节点数量)
        column_indices = [unique_nodes[n] for samp_neigh in samp_neighs for n in samp_neigh]  # 保存列 每一行对应的邻居真实index做为列。
        row_indices = [i for i in range(len(samp_neighs)) for j in range(len(samp_neighs[i]))]# 保存行 每行邻居数
        mask[row_indices, column_indices] = 1
        num_neigh = mask.sum(1, keepdim=True)                         # 按行求和，保持和输入一个维度
        mask = mask.div(num_neigh).to(embed_matrix.device)            # 归一化操作
        aggregate_feats = mask.mm(embed_matrix)                       # 矩阵相乘，相当于聚合周围邻接信息求和

        return aggregate_feats

In [5]:
import math
import tqdm
from sklearn.metrics import roc_auc_score

torch.manual_seed(0)
def train_one_epoch(data, graphsage, batch_size, device, optimizer):
    graphsage.train()

    train_idx = data.train_mask
    perm = torch.randperm(train_idx.size(0), device=device)
    train_idx = train_idx[perm]
    labels = data.y.to(device)

    total_loss = 0.0
    batches = math.ceil(len(train_idx) / batch_size)
    pbar = tqdm.trange(1, batches + 1, desc="Training", ncols=100)

    for index in pbar:
        start = (index - 1) * batch_size
        end = index * batch_size
        nodes_batch = train_idx[start:end]

        labels_batch = labels[nodes_batch]
        logits = graphsage(nodes_batch)

        loss = F.nll_loss(logits, labels_batch)
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(list(graphsage.parameters()) , 5)
        optimizer.step()
        avg_loss = total_loss / index
        pbar.set_postfix(
            {
                "loss": f"{loss.item():.4f}",
                "avg_loss": f"{avg_loss:.4f}",
            })

    return total_loss / batches


def train(data, model, batch_size, device, epoches, name="graphsage"):
    max_val_auc = 0.0
    optimizer = torch.optim.Adam(
        list(model.parameters()), lr=5e-4
    )

    for epoch in range(1, epoches + 1):
        print(f"\n===== Epoch {epoch} =====")
        loss = train_one_epoch(data, model, batch_size, device, optimizer)
        print(f"Train Loss: {loss:.4f}")
        max_val_auc = evaluate(data, model, batch_size, device, max_val_auc, name, epoch)

    print(f"Best Validation AUC: {max_val_auc:.4f}")

In [6]:
import torch
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate(data, model, batch_size, device, max_val_auc, name, cur_epoch):
    model.eval()

    x, y = data.x.to(device), data.y.to(device)
    val_idx = data.valid_mask
    test_idx = data.test_mask

    # === 验证集 ===
    val_probs = torch.zeros(len(val_idx), device=device)
    for i in tqdm.tqdm(range(0, len(val_idx), batch_size), desc=f"Validating Epoch {cur_epoch}"):
        batch = val_idx[i:i+batch_size]
        logits = model(batch)
        val_probs[i:i+batch_size] = logits.exp()[:, 1]

    val_labels = y[val_idx]
    val_auc = roc_auc_score(val_labels.cpu(), val_probs.cpu())
    print(f"Validation AUC: {val_auc:.4f}")

    # === 若更优，再测试 ===
    if val_auc > max_val_auc:
        torch.save(
            model.state_dict(),
            f'./results/model_best_{name}_ep{cur_epoch}_{val_auc:.4f}.pt'
        )
        max_val_auc = val_auc

    return max_val_auc


In [10]:
torch.manual_seed(0)
num_labels = 2
batch_size = 100
device = "cuda:0" if torch.cuda.is_available() else "cpu"
feature_size = 20
epoches = 100
hidden_size = 64
num_layers = 2 # 采样邻居depth，论文中的B

data = data.to(device)
model = GraphSage(
    num_layers, 
    data.x.size(-1),
    hidden_size,
    data.x,
    data.adj_t,
    device
).to(device)
print("总参数量: ")
print(sum(p.numel() for p in model.parameters()))

# train(data, model, batch_size, device, 100)

总参数量: 
10882


In [12]:
## 生成 main.py 时请勾选此 cell
from utils import DGraphFin
from utils.evaluator import Evaluator
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch_geometric.transforms as T
from torch_geometric.data import Data
import numpy as np
import os

# 这里可以加载你的模型
model.load_state_dict(
    torch.load('./results/small/model_best_graphsage_ep21_0.7700.pt'))

def predict(data,node_id):
    """
    加载模型和模型预测
    :param node_id: int, 需要进行预测节点的下标
    :return: tensor, 类0以及类1的概率, torch.size[1,2]
    """
    
    # 模型预测时，测试数据已经进行了归一化处理
    # -------------------------- 实现模型预测部分的代码 ---------------------------
    with torch.no_grad():
        model.eval()
        out = model([node_id])
        y_pred = out.exp()  # (N,num_classes)
        
    return y_pred.squeeze(0)

pred = predict(data, 0)
print(pred.shape)
print(pred)


torch.Size([2])
tensor([0.9870, 0.0130])
